In [49]:
import os, time, math
from pathlib import Path
import requests
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
import unicodedata, ast
import numpy as np, ast
import re

In [50]:
# Carga API key desde env (TMDB_API_KEY=tu_clave)

load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_API_KEY")
assert TMDB_API_KEY, "Falta variable de entorno TMDB_API_KEY"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True, parents=True)

BASE_URL = "https://api.themoviedb.org/3"

# Sesión HTTP con retry
session = requests.Session()
session.params = {"api_key": TMDB_API_KEY}

def tmdb_get(path, params=None, lang=None, retries=3, sleep=0.25):
    """
    Llamada a TMDb con manejo básico de rate-limit/errores.
    - path: e.g. '/discover/movie'
    - lang: 'es-ES' o 'en-US' (opcional)
    """
    url = f"{BASE_URL}{path}"
    params = dict(params or {})
    if lang:
        params["language"] = lang

    for i in range(retries):
        resp = session.get(url, params=params, timeout=30)
        if resp.status_code == 200:
            return resp.json()
        # 429: rate limit
        if resp.status_code == 429:
            time.sleep(1.0 + i)
            continue
        # 5xx: reintenta suave
        if 500 <= resp.status_code < 600:
            time.sleep(0.5 + i*0.5)
            continue
        # Otros errores: levanta
        resp.raise_for_status()
    raise RuntimeError(f"TMDb fallo tras {retries} intentos: {path} {params}")

In [51]:
def discover_movie_ids_by_year(year, max_pages=None):
    """
    Descubre películas por año con filtros de calidad y respeta el límite de 500 páginas de TMDb.
    """
    items = []
    page = 1
    while True:
        params = {
            "sort_by": "popularity.desc",
            "include_adult": "false",
            "include_video": "false",
            "primary_release_date.gte": f"{year}-01-01",
            "primary_release_date.lte": f"{year}-12-31",
            # Filtros de calidad para reducir páginas
            "vote_count.gte": 200,
            "vote_average.gte": 6.0,
            "page": page,
        }
        data = tmdb_get("/discover/movie", params=params, lang="es-ES")
        results = data.get("results", [])
        items.extend(results)

        total_pages = min(data.get("total_pages", 1), 500)
        if max_pages:
            total_pages = min(total_pages, max_pages)

        if page >= total_pages:
            break

        page += 1
        time.sleep(0.15)

    return items

# Prueba rápida: trae 2023-2024 (2 páginas por año para muestra)
years_sample = [2023, 2024]
raw_items = []
for y in tqdm(years_sample, desc="Discover (muestra)"):
    raw_items.extend(discover_movie_ids_by_year(y, max_pages=2))

len(raw_items), raw_items[0].get("id") if raw_items else None


Discover (muestra): 100%|██████████| 2/2 [00:01<00:00,  1.93it/s]


(80, 1010581)

In [66]:
# =========================
# Enriquecimiento TMDb: posters + overview
# =========================

IMAGE_BASE = "https://image.tmdb.org/t/p"  # TMDb secure base fijo para MVP

def build_image_url(path: str | None, size: str = "w342") -> str | None:
    """Construye URL completa para poster/backdrop. Devuelve None si no hay path."""
    if not path or not isinstance(path, str):
        return None
    # tamaños útiles: posters -> w342 / w500; backdrops -> w780
    return f"{IMAGE_BASE}/{size}{path}"

def _safe_list(x, key="name"):
    """Convierte listas de dicts a lista de strings (p. ej., genres/keywords)."""
    if isinstance(x, list):
        out = []
        for item in x:
            if isinstance(item, dict) and key in item:
                out.append(str(item[key]).strip())
            elif isinstance(item, str):
                out.append(item.strip())
        return [t for t in out if t]
    return []

def get_movie_details_enriched(movie_id: int, lang_primary="es-ES", lang_fallback="en-US") -> dict:
    """
    Detalles enriquecidos con 'append_to_response' para traer keywords en un tiro.
    Fallback de idioma para overview si viene vacío en español.
    """
    # 1) Intento en español
    d_es = tmdb_get(f"/movie/{movie_id}", params={"append_to_response": "keywords"}, lang=lang_primary)

    # 2) Si overview vacío o muy corto, pide en inglés y fusiona solo lo faltante
    overview = (d_es.get("overview") or "").strip()
    d_en = None
    if len(overview) < 10 and lang_fallback:
        d_en = tmdb_get(f"/movie/{movie_id}", params={"append_to_response": "keywords"}, lang=lang_fallback)

    # 3) Campos base
    release_date = d_es.get("release_date") or (d_en.get("release_date") if d_en else None)
    release_year = int(release_date.split("-")[0]) if isinstance(release_date, str) and len(release_date) >= 4 else None

    genres = _safe_list(d_es.get("genres")) or (_safe_list(d_en.get("genres")) if d_en else [])
    kw_block = d_es.get("keywords") or (d_en.get("keywords") if d_en else {})
    keywords = _safe_list(kw_block.get("keywords") or kw_block.get("results") or [], key="name")

    poster_path = d_es.get("poster_path") or (d_en.get("poster_path") if d_en else None)
    backdrop_path = d_es.get("backdrop_path") or (d_en.get("backdrop_path") if d_en else None)

    # 4) Fallback de overview (solo si el español no sirvió)
    if len(overview) < 10 and d_en:
        overview = (d_en.get("overview") or "").strip()

    # 5) Construye URLs listas para UI
    poster_url_w342 = build_image_url(poster_path, "w342")
    poster_url_w500 = build_image_url(poster_path, "w500")
    backdrop_url_w780 = build_image_url(backdrop_path, "w780")

    # 6) Arma el dict fila compatible con tu catálogo
    row = {
        "id": d_es.get("id") or (d_en.get("id") if d_en else movie_id),
        "title": d_es.get("title") or (d_en.get("title") if d_en else None),
        "original_title": d_es.get("original_title") or (d_en.get("original_title") if d_en else None),
        "original_language": d_es.get("original_language") or (d_en.get("original_language") if d_en else None),
        "release_date": release_date,
        "release_year": release_year,
        "runtime": d_es.get("runtime") or (d_en.get("runtime") if d_en else None),
        "vote_average": d_es.get("vote_average"),
        "vote_count": d_es.get("vote_count"),
        "popularity": d_es.get("popularity"),
        "genres": genres,                # lista de strings
        "keywords": keywords,            # lista de strings
        "belongs_to_collection": d_es.get("belongs_to_collection") or (d_en.get("belongs_to_collection") if d_en else None),
        "poster_path": poster_path,
        "poster_url_w342": poster_url_w342,
        "poster_url_w500": poster_url_w500,
        "backdrop_path": backdrop_path,
        "backdrop_url_w780": backdrop_url_w780,
        "overview": overview,
    }
    return row

# Enriquecer SOLO una muestra (primeras 60 pelis) para validar el flujo
sample_ids = list({it["id"] for it in raw_items})[:60]
rows = []
for mid in tqdm(sample_ids, desc="Detalles+Keywords (muestra)"):
    try:
        rows.append(get_movie_details_enriched(mid))
    except Exception as e:
        print("Error con id", mid, e)
    time.sleep(0.15) 

df_sample = pd.DataFrame(rows)
df_sample.head(3)


Detalles+Keywords (muestra): 100%|██████████| 60/60 [00:10<00:00,  5.61it/s]


,id,title,original_title,original_language,release_date,release_year,runtime,vote_average,vote_count,popularity,genres,keywords,belongs_to_collection,poster_path,poster_url_w342,poster_url_w500,backdrop_path,backdrop_url_w780,overview
0,940551,Migración. Un viaje patas arriba,Migration,en,2023-12-06,2023,82,7.383,2113,12.7539,"[Familia, Comedia, Aventura, Animación]","[duck, villain, migration, flight, anthropomor...",None,/diEeiB2DmZZadHISkg24RO2n0rT.jpg,https://image.tmdb.org/t/p/w342/diEeiB2DmZZadH...,https://image.tmdb.org/t/p/w500/diEeiB2DmZZadH...,/gklkxY0veMajdCiGe6ggsh07VG2.jpg,https://image.tmdb.org/t/p/w780/gklkxY0veMajdC...,La familia Mallard se ha quedado 'estancada'. ...
1,519182,Gru 4. Mi villano favorito,Despicable Me 4,en,2024-06-20,2024,94,7.000,2958,29.1751,"[Familia, Comedia, Animación, Ciencia ficción]","[superhero, villain, sequel, super villain, af...","{'id': 86066, 'name': 'Gru, mi villano favorit...",/b6JX0fBne5yPFNBtdp4Imi3CpiE.jpg,https://image.tmdb.org/t/p/w342/b6JX0fBne5yPFN...,https://image.tmdb.org/t/p/w500/b6JX0fBne5yPFN...,/twsxsfao6ZOVvT8LfudH603MMi6.jpg,https://image.tmdb.org/t/p/w780/twsxsfao6ZOVvT...,"Gru, Lucy y las niñas -Margo, Edith y Agnes- d..."
2,1079310,La receta perfecta,Vingt Dieux,fr,2024-12-11,2024,92,7.013,271,13.1698,"[Drama, Comedia]","[cheese, complicated birth, stock car racing, ...",None,/bx9QYAHvxpxYcalWQZjT4IJ8h1O.jpg,https://image.tmdb.org/t/p/w342/bx9QYAHvxpxYca...,https://image.tmdb.org/t/p/w500/bx9QYAHvxpxYca...,/AllcXFGjhy3NbZSLCnb55LFlNQo.jpg,https://image.tmdb.org/t/p/w780/AllcXFGjhy3NbZ...,"Totone, de 18 años, pasa la mayor parte del ti..."


In [67]:
def quality_filter(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()
    df2 = df2[df2["vote_count"].fillna(0) >= 200]
    df2 = df2[df2["vote_average"].fillna(0) >= 6.0]
    # df2 = df2[(df2["runtime"].fillna(0) >= 60) & (df2["runtime"].fillna(0) <= 240)]
    if "overview" in df2.columns:
        df2["overview"] = df2["overview"].astype(str)
        df2 = df2[df2["overview"].str.len() > 0]
    return df2

In [69]:
def derive_saga_key(row) -> str:
    col = row.get("belongs_to_collection")
    if isinstance(col, dict) and col.get("name"):
        return str(col["name"]).strip().lower()
    # Heurística simple por título (opcional)
    t = str(row.get("title") or "").lower()
    for token in [" ii", " iii", " iv", " v ", " 2", " 3", " 4"]:
        if token in t:
            return re.sub(r"[:\-–].*$", "", t).strip()
    return t  # fallback: título mismo (no perfecto, pero útil)

if "saga_key" not in df.columns:
    df["saga_key"] = df.apply(derive_saga_key, axis=1)

In [70]:
sample_ids = list({it["id"] for it in raw_items})[:60]
rows = []
for mid in tqdm(sample_ids, desc="Detalles+Keywords (muestra)"):
    try:
        rows.append(get_movie_details_enriched(mid))
    except Exception as e:
        print("Error con id", mid, e)
    time.sleep(0.15)

df_sample = pd.DataFrame(rows)
df_sample["saga_key"] = df_sample.apply(derive_saga_key, axis=1)
df_ok = quality_filter(df_sample).reset_index(drop=True)
print("Filas totales (muestra):", len(df_sample), " | Tras filtro:", len(df_ok))
df_ok[["title","release_year","poster_url_w342","overview"]].head(5)


Detalles+Keywords (muestra): 100%|██████████| 60/60 [00:10<00:00,  5.52it/s]

Filas totales (muestra): 60  | Tras filtro: 60


,title,release_year,poster_url_w342,overview
0,Migración. Un viaje patas arriba,2023,https://image.tmdb.org/t/p/w342/diEeiB2DmZZadH...,La familia Mallard se ha quedado 'estancada'. ...
1,Gru 4. Mi villano favorito,2024,https://image.tmdb.org/t/p/w342/b6JX0fBne5yPFN...,"Gru, Lucy y las niñas -Margo, Edith y Agnes- d..."
2,La receta perfecta,2024,https://image.tmdb.org/t/p/w342/bx9QYAHvxpxYca...,"Totone, de 18 años, pasa la mayor parte del ti..."
3,Ninja Turtles: Caos mutante,2023,https://image.tmdb.org/t/p/w342/mgBXgA8jHext4K...,Después de pasar años apartados del mundo huma...
4,Miraculous World: Las Aventuras de Ladybug en ...,2024,https://image.tmdb.org/t/p/w342/gdZ4REoGYhUk4O...,"Para salvar el futuro de un terrible destino, ..."


In [72]:
years_full = list(range(2000, 2026))
all_ids = set()

for y in tqdm(years_full, desc="Discover (full)"):
    its = discover_movie_ids_by_year(y)
    all_ids.update(it["id"] for it in its)

print("IDs únicos:", len(all_ids))

rows = []
checkpoint_every = 500  # guarda cada 500 por si se corta
for i, mid in enumerate(tqdm(list(all_ids), desc="Detalles+Keywords (full)"), start=1):
    try:
        rows.append(get_movie_details_enriched(mid))
    except Exception as e:
        # log suave y continúa
        # print("Error con id", mid, e)
        pass
    time.sleep(0.18)  # cuida rate limit

    # checkpoint
    if i % checkpoint_every == 0:
        df_tmp = pd.DataFrame(rows).drop_duplicates(subset=["id"])
        df_tmp["saga_key"] = df_tmp.apply(derive_saga_key, axis=1)
        df_ck = quality_filter(df_tmp).reset_index(drop=True)
        df_ck.to_parquet(DATA_DIR / f"catalogo_ck_{i}.parquet", index=False)

# build final
df_full = pd.DataFrame(rows).drop_duplicates(subset=["id"]).reset_index(drop=True)
df_full["saga_key"] = df_full.apply(derive_saga_key, axis=1)
df_full_ok = quality_filter(df_full).reset_index(drop=True)

# columnas importantes presentes (no reordena, solo asegura)
need_cols = ["poster_url_w342","overview","saga_key","genres","keywords","release_year","title"]
missing = [c for c in need_cols if c not in df_full_ok.columns]
if missing:
    print("⚠️ Faltan columnas no críticas:", missing)

# guarda catálogo final
out_path = DATA_DIR / "catalogo_peliculas.parquet"
df_full_ok.to_parquet(out_path, index=False)
len(df_full), len(df_full_ok), out_path


Discover (full): 100%|██████████| 26/26 [01:33<00:00,  3.59s/it]


IDs únicos: 7351


Detalles+Keywords (full): 100%|██████████| 7351/7351 [37:01<00:00,  3.31it/s]  


(7351, 7350, PosixPath('data/catalogo_peliculas.parquet'))

In [73]:
#mostrar los dataframes

print(df_full.sample(10))

          id                               title  \
530     1599              Extrañas coincidencias   
4359  900667                  One Piece Film Red   
6983  849869                  Boksoon debe morir   
1374  726139                     Project Silence   
3583  799583    El ministerio de la Guerra Sucia   
6640  454433                          Magic Camp   
7137  293646  Los 33 (Una Historia De Esperanza)   
6597  912916                        La otra Zoey   
6355  419831                       I Kill Giants   
6916  980477                            Ne Zha 2   

                             original_title original_language release_date  \
530                           I ♥ Huckabees                en   2004-09-10   
4359                     ONE PIECE FILM RED                ja   2022-08-06   
6983                                    길복순                ko   2023-02-17   
1374                          탈출: 프로젝트 사일런스                ko   2024-07-11   
3583  The Ministry of Ungentlemanly W

In [74]:
print(df_full_ok.sample(10))

           id                           title                original_title  \
6774   127585   X-Men: Días del futuro pasado    X-Men: Days of Future Past   
3850   439998           Omicidio all'italiana         Omicidio all'italiana   
3609    13223                     Gran Torino                   Gran Torino   
3620  1160164    TAYLOR SWIFT | THE ERAS TOUR  TAYLOR SWIFT | THE ERAS TOUR   
5420   611291  BTS: Bring the Soul: The Movie                 브링 더 소울: 더 무비   
913    265195                Relatos salvajes              Relatos salvajes   
5281   381289                  Tu mejor amigo               A Dog's Purpose   
6470   486070                  Bendita locura              Benedetta follia   
6882  1045770                     Daaaaaalí !                   Daaaaaalí !   
6712   258284           Secretos de un crimen            Every Secret Thing   

     original_language release_date  release_year  runtime  vote_average  \
6774                en   2014-05-15          2014     

In [76]:
# leer el fichero parquet

df = pd.read_parquet("data/catalogo_peliculas.parquet")
df.shape, df.columns.tolist()[:10]

((7350, 20),
 ['id',
  'title',
  'original_title',
  'original_language',
  'release_date',
  'release_year',
  'runtime',
  'vote_average',
  'vote_count',
  'popularity'])

In [34]:
df

,tmdb_id,title,overview_es,overview_en,overview,genres,keywords,release_date,release_year,runtime,vote_average,vote_count,popularity,poster_path,original_language,production_countries
0,524288,Una revolución en toda regla,"En un pueblo rural de Delhi (India), las mujer...",None,"En un pueblo rural de Delhi (India), las mujer...",[Documental],[short film],2018-04-05,2018,26,7.797,244,0.5956,/r1TKbtynWG4haAda4UwMBByPpSO.jpg,hi,[IN]
1,655363,Post Mortem: Fotos del Más Allá (Después de la...,"En el frío invierno de 1918, Tomás, un joven q...",None,"En el frío invierno de 1918, Tomás, un joven q...","[Terror, Misterio, Suspense]",[paranormal investigation],2020-10-28,2020,115,6.720,225,3.1645,/15fqMMg5s72NJHYRfO6lshslOHw.jpg,hu,[HU]
2,12,Buscando a Nemo,"Nemo, un pececillo, hijo único muy querido y p...",None,"Nemo, un pececillo, hijo único muy querido y p...","[Animación, Familia]","[fish, sydney, australia, parent child relatio...",2003-05-30,2003,101,7.816,19972,13.6676,/jPhak722pNGxQIXSEfeWIUqBrO5.jpg,en,[US]
3,16,Bailar en la oscuridad,La película se desarrolla en Estados Unidos en...,None,La película se desarrolla en Estados Unidos en...,"[Drama, Crimen]","[factory worker, individual, immigrant, blindn...",2000-09-01,2000,140,7.863,1897,5.0084,/DibSnXZBioqf8NngFlrAlmzVs.jpg,en,"[DK, FI, FR, DE, IS, NL, NO, SE]"
4,20,Mi vida sin mí,"Ann tiene 23 años, dos hijas, un marido que pa...",None,"Ann tiene 23 años, dos hijas, un marido que pa...","[Drama, Romance]","[dying and death, daughter, farewell, night sh...",2003-03-07,2003,106,6.063,481,2.7186,/uxm1RCFWTlROoLojSiP0HmFq8be.jpg,en,"[CA, ES]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7338,32740,Krrish,Vestido con un elegante atuendo negro y el ros...,None,Vestido con un elegante atuendo negro y el ros...,"[Acción, Ciencia ficción]",[superhero],2006-06-23,2006,185,6.414,251,3.3314,/neJo0Xt9NH6aPBPNhKfHFQpwrcC.jpg,hi,"[HK, IN]"
7339,458737,El reverendo,El encuentro con un activista y su esposa emba...,None,El encuentro con un activista y su esposa emba...,[Drama],"[suicide, funeral, explosive, faith, christian...",2018-05-18,2018,113,7.000,1394,2.6394,/vwGXSZAUPguB53fbgm53aUg2Zus.jpg,en,"[US, GB]"
7340,425972,Cargo,Cargo nos lleva hasta un mundo azotado por un ...,None,Cargo nos lleva hasta un mundo azotado por un ...,"[Drama, Suspense, Terror]","[australia, post-apocalyptic future, woman dir...",2017-10-06,2017,105,6.403,1792,3.6929,/cdPSUck4tBRvRu6DFk6XciDrssn.jpg,en,"[AU, GB, US]"
7341,917496,Bitelchús Bitelchús,"Tras una inesperada tragedia familiar, tres ge...",None,"Tras una inesperada tragedia familiar, tres ge...","[Comedia, Fantasía, Terror]","[afterlife, haunted house, sequel, paranormal,...",2024-09-04,2024,105,7.000,2788,12.0076,/kWJw7dCWHcfMLr0irTHAPIKrJ4I.jpg,en,[US]


In [112]:
# ---------- Utilidades ----------

def _ensure_list(x):
    """
    Convierte x a lista sin caer en el error 'truth value of an array is ambiguous'.
    Soporta: list, tuple, set, numpy.ndarray, dict{'name':...}, strings 'a,b' o '["a","b"]',
    None/NaN y otros tipos raros.
    """
    # 1) contenedores comunes
    if isinstance(x, list):
        return x
    if isinstance(x, (tuple, set)):
        return list(x)

    # 2) numpy arrays
    try:
        import numpy as np
        if isinstance(x, np.ndarray):
            return x.tolist()
    except Exception:
        pass

    # 3) nulos
    if x is None:
        return []
    try:
        # solo evalúa isna en escalares, no en arrays
        if isinstance(x, (float, int)) and pd.isna(x):
            return []
    except Exception:
        pass

    # 4) dicts típicos de TMDB (por si se coló uno suelto)
    if isinstance(x, dict):
        if "name" in x and isinstance(x["name"], str):
            return [x["name"].strip()]
        return [str(x)]

    # 5) strings: intenta literal list -> si no, separa por comas
    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            import ast
            try:
                val = ast.literal_eval(s)
                if isinstance(val, (list, tuple, set)):
                    return [str(v).strip() for v in val if str(v).strip()]
            except Exception:
                pass
        return [p.strip() for p in s.split(",") if p.strip()]

    # 6) fallback
    return [x]

def has_any(values, target_set):
    values = _ensure_list(values)
    return any(v in target_set for v in values)

def minmax_norm(s, clip=True):
    s = s.astype(float)
    if s.max() == s.min():
        return pd.Series(np.ones(len(s))*0.5, index=s.index)
    z = (s - s.min()) / (s.max() - s.min())
    return z.clip(0,1) if clip else z

def recency_score(year, base_year=2000, floor=1980):
    # Años más recientes = mayor puntaje (0-1). 
    y = year.fillna(base_year).astype(int)
    y = y.clip(lower=floor)  # evita negativos
    return minmax_norm(y)

def votes_confidence(vote_count, k=200): 
    # Suavizado simple: votos/(votos+k) -> 0-1; más votos = más confianza
    v = vote_count.fillna(0).astype(float)
    return (v / (v + k)).clip(0,1)

def rating_shrunk(vote_average, vote_count, global_mean=6.8, k=200):
    # Bayesian shrinkage sencillo para no sobrevalorar ratings con pocos votos
    r = vote_average.fillna(global_mean).astype(float)
    v = vote_count.fillna(0).astype(float)
    return ((v/(v+k))*r + (k/(v+k))*global_mean)

def derive_saga_key(row):
    # Prioridad: belongs_to_collection.name -> si no existe, heurística por título sin números finales
    if 'belongs_to_collection' in row and isinstance(row['belongs_to_collection'], dict):
        name = row['belongs_to_collection'].get('name')
        if isinstance(name, str) and name.strip():
            return name.strip().lower()
    title = (row.get('title') or "").strip().lower()
    # Heurística básica: quita números romanos/árabes al final (rocky ii -> rocky)
    base = title
    base = base.replace(" part ", " ")
    base = base.replace(" capítulo ", " ")
    base = base.replace(" capitulo ", " ")
    base = base.rsplit(" ", 1)[0] if base.split()[-1].isdigit() else base
    return base

def diversify_by_group(df, group_col, top_n=10, per_group_max=1):
    if group_col not in df.columns:
        return df.sort_values(["score_total","pop_norm","title"], ascending=[False,False,True]).head(top_n)
    # ordena por score y luego toma a lo mucho N por grupo
    ordered = df.sort_values(["score_total","pop_norm","title"], ascending=[False,False,True]).copy()
    ordered["_rank_in_group"] = ordered.groupby(group_col).cumcount()
    diverse = ordered[ordered["_rank_in_group"] < per_group_max]
    return diverse.drop(columns=["_rank_in_group"]).head(top_n)

In [113]:
# ---------- Núcleo del pipeline ----------
def rank_mood(
    df_raw,
    genre_sets=None,         # {"must_any":[set,..], "boost_any":[set,..]}
    keyword_sets=None,       # {"must_any":[set,..], "boost_any":[set,..]}
    weights=None,            # dict con pesos de cada componente
    filters=None,            # dict con mínimos/máximos (runtime, votes, year, etc.)
    top_n=None,              # None = sin límite, int = recorte opcional
    diversify_by="saga_key",
    per_group_max=None,      # None = sin límite, int = máx. por grupo
    explain=True,
):
    """
    Evalúa y rankea películas para un 'mood' dado, devolviendo todo el universo posible.
    Sin recortes salvo que se especifiquen manualmente (top_n o per_group_max).
    """
    df = df_raw.copy()

    # ---- Limpieza y coerción ----
    for col in ["genres", "keywords"]:
        if col in df.columns:
            df[col] = df[col].apply(_ensure_list)
        else:
            df[col] = [[] for _ in range(len(df))]
    for col in ["release_year", "vote_average", "vote_count", "popularity", "runtime"]:
        if col not in df.columns:
            df[col] = np.nan

    # Generar saga_key si hace falta
    if diversify_by and diversify_by not in df.columns:
        df[diversify_by] = df.apply(derive_saga_key, axis=1)

    # ---- Filtros ----
    filters = filters or {}
    df_cand = df.copy()

    yr_min, yr_max = filters.get("year_min"), filters.get("year_max")
    rt_min, rt_max = filters.get("runtime_min"), filters.get("runtime_max")
    vc_min = filters.get("min_votes")

    if yr_min is not None:
        df_cand = df_cand[df_cand["release_year"].fillna(0) >= yr_min]
    if yr_max is not None:
        df_cand = df_cand[df_cand["release_year"].fillna(9999) <= yr_max]
    if rt_min is not None:
        df_cand = df_cand[df_cand["runtime"].fillna(0) >= rt_min]
    if rt_max is not None:
        df_cand = df_cand[df_cand["runtime"].fillna(1e9) <= rt_max]
    if vc_min is not None:
        df_cand = df_cand[df_cand["vote_count"].fillna(0) >= vc_min]

    if len(df_cand) == 0:
        return df_cand  # vacío si no hay matches

    # ---- Matching por géneros y keywords ----
    genre_sets = genre_sets or {}
    keyword_sets = keyword_sets or {}

    def any_from_anyset(vals, sets_list):
        if not sets_list:
            return False
        return any(has_any(vals, s) for s in sets_list)

    must_gen_ok = df_cand["genres"].apply(lambda g: any_from_anyset(g, genre_sets.get("must_any", [])))
    must_kw_ok = df_cand["keywords"].apply(lambda k: any_from_anyset(k, keyword_sets.get("must_any", [])))

    if genre_sets.get("must_any"):
        df_cand = df_cand[must_gen_ok]
    if keyword_sets.get("must_any"):
        df_cand = df_cand[must_kw_ok]
    if len(df_cand) == 0:
        return df_cand

    # ---- Scoring ----
    gen_boost = df_cand["genres"].apply(lambda g: sum(int(has_any(g, s)) for s in genre_sets.get("boost_any", [])))
    kw_boost = df_cand["keywords"].apply(lambda k: sum(int(has_any(k, s)) for s in keyword_sets.get("boost_any", [])))
    match_raw = gen_boost + kw_boost

    rec = recency_score(df_cand["release_year"])
    rating_adj = rating_shrunk(df_cand["vote_average"], df_cand["vote_count"])
    rating_norm = minmax_norm(pd.Series(rating_adj, index=df_cand.index))
    votes_norm = votes_confidence(df_cand["vote_count"], k=filters.get("k_votes", 200))
    pop_norm = minmax_norm(df_cand["popularity"].fillna(0))
    match_norm = minmax_norm(match_raw)

    W = {"w_match": 0.40, "w_recency": 0.10, "w_rating": 0.25, "w_votes": 0.10, "w_pop": 0.15}
    if weights:
        W.update(weights)

    df_cand["match_raw"] = match_raw
    df_cand["match_norm"] = match_norm
    df_cand["recencia"] = rec
    df_cand["rating_norm"] = rating_norm
    df_cand["votes_norm"] = votes_norm
    df_cand["pop_norm"] = pop_norm
    df_cand["score_total"] = (
        W["w_match"] * match_norm +
        W["w_recency"] * rec +
        W["w_rating"] * rating_norm +
        W["w_votes"] * votes_norm +
        W["w_pop"] * pop_norm
    )

    # ---- Diversificación ----
    df_cand = df_cand.sort_values("score_total", ascending=False)
    if diversify_by and diversify_by in df_cand.columns and per_group_max:
        df_cand = df_cand.groupby(diversify_by, group_keys=False).head(per_group_max)

    # ---- Explicaciones ----
    if explain:
        df_cand["explicacion"] = df_cand.apply(lambda r: make_reason_row(r, "unknown"), axis=1)
    else:
        df_cand["explicacion"] = None

    # ---- Ranking dentro del mood ----
    df_cand["rank_within_mood"] = range(1, len(df_cand) + 1)

    # ---- Aplicar top_n opcional ----
    if isinstance(top_n, int) and top_n > 0:
        df_cand = df_cand.head(top_n)

    # ---- Selección final ----
    base_cols = [
        "title", "release_year", "genres", "runtime", "vote_average", "vote_count",
        "popularity", "poster_url_w342", "overview"
    ]
    extra_cols = [diversify_by] if diversify_by and diversify_by in df_cand.columns else []
    expl_cols = ["match_raw", "match_norm", "recencia", "rating_norm", "votes_norm", "pop_norm", "score_total", "explicacion", "rank_within_mood"]
    cols = [c for c in base_cols if c in df_cand.columns] + extra_cols + expl_cols
    return df_cand[cols].reset_index(drop=True)


# ---------- Build para los moods + consolidación ----------
def build_all_moods_tops(df_catalogo, top_n=None, diversify_by="saga_key", per_group_max=None, explain=True):
    cfgs = mood_configs()
    frames = []
    for mood, cfg in cfgs.items():
        df_ranked = rank_mood(
            df_raw=df_catalogo,
            genre_sets=cfg.get("genre_sets"),
            keyword_sets=cfg.get("keyword_sets"),
            weights=WEIGHTS.get(mood),
            filters=FILTERS.get(mood),
            top_n=top_n,                    # ahora puede ser None
            diversify_by=diversify_by,
            per_group_max=per_group_max,    # ahora puede ser None
            explain=explain
        )
        df_ranked["mood"] = mood
        frames.append(df_ranked)

    df_all = pd.concat(frames, ignore_index=True)
    df_all = add_explanations_to_df(df_all)

    order_cols = [c for c in ["score_total", "pop_norm", "title"] if c in df_all.columns]
    asc = [False if c != "title" else True for c in order_cols]
    return df_all.sort_values(order_cols, ascending=asc).reset_index(drop=True)


# ---------- Ejecutar y guardar ----------
catalog_path = DATA_DIR / "catalogo_peliculas.parquet"
df_catalogo = pd.read_parquet(catalog_path)

if "saga_key" not in df_catalogo.columns:
    df_catalogo["saga_key"] = df_catalogo.apply(derive_saga_key, axis=1)

# No topes: guarda TODO el universo filtrado y diversificado
df_tops = build_all_moods_tops(
    df_catalogo,
    top_n=None,              # None = sin límite
    diversify_by="saga_key",
    per_group_max=None,      # None = sin límite
    explain=True
)

needed = ["poster_url_w342", "overview", "saga_key", "mood", "score_total", "explicacion", "title", "release_year"]
missing = [c for c in needed if c not in df_tops.columns]
if missing:
    print("Faltan columnas en tops:", missing)

tops_path = DATA_DIR / "tops.parquet"
df_tops.to_parquet(tops_path, index=False)
print("✅ Saved tops:", tops_path, "| filas:", len(df_tops))

✅ Saved tops: data/tops.parquet | filas: 13956


In [114]:
# =========================
# CONSOLIDADO + PERSISTENCIA + QA  (soporta dict o DataFrame)
# =========================

TOP_K = None  # None -> sin tope; 200 -> guarda top 200 por mood

def _ensure_listlike(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    # géneros/keywords guardados como string JSON/CSV → intenta normalizar
    try:
        import json
        j = json.loads(x)
        return j if isinstance(j, list) else [j]
    except Exception:
        # fallback: separa por coma
        return [t.strip() for t in str(x).split(",") if str(x).strip()]

def consolidate_and_persist(mood_frames, out_path="data/tops.parquet", dedup_saga_for_QA=False):
    """
    mood_frames: dict[str, pd.DataFrame] o pd.DataFrame ya con columna 'mood'
    out_path: ruta de salida parquet
    dedup_saga_for_QA: si True, imprime un sample '1 por saga' para cada mood (solo para QA),
                      pero el archivo guardado siempre incluye TODO el universo.
    """

    if isinstance(mood_frames, dict):
        parts = []
        for mood, dfm in mood_frames.items():
            if dfm is None or len(dfm) == 0:
                continue
            dfm = dfm.copy()

            # Orden por score_total DESC
            if "score_total" in dfm.columns:
                dfm = dfm.sort_values("score_total", ascending=False)

            # Rank dentro de cada mood (1 = mejor)
            dfm["mood"] = mood
            dfm["rank_within_mood"] = range(1, len(dfm) + 1)

            # Tope opcional (solo si TOP_K tiene número)
            if isinstance(TOP_K, int) and TOP_K > 0:
                dfm = dfm.head(TOP_K)

            parts.append(dfm)

        if not parts:
            raise ValueError("No hay data consolidable. Revisa los dataframes por mood.")
        full = pd.concat(parts, ignore_index=True)
    else:
        # Ya venía consolidado
        full = mood_frames.copy()
        if "score_total" in full.columns:
            # reordenar y rankear por seguridad
            full = full.sort_values(["mood", "score_total"], ascending=[True, False])
            full["rank_within_mood"] = full.groupby("mood").cumcount() + 1
        if isinstance(TOP_K, int) and TOP_K > 0:
            full = full.groupby("mood", group_keys=False).head(TOP_K)

    # Normaliza algunos campos útiles (evita strings raros)
    for col in ["genres", "keywords_match", "keywords_boost"]:
        if col in full.columns:
            full[col] = full[col].apply(_ensure_listlike)

    # Columnas esperadas por la app (ajusta según tu schema real)
    expected = [
        "title", "release_year", "runtime", "genres",
        "vote_average", "vote_count", "popularity",
        "match_raw", "score_total", "saga_key",
        "overview", "poster_url_w342", "explicacion", "mood",
        "rank_within_mood"
    ]
    # agrega columnas faltantes si fuera necesario
    for c in expected:
        if c not in full.columns:
            full[c] = None

    # Orden final (mood, rank)
    if "score_total" in full.columns:
        full = full.sort_values(["mood", "score_total"], ascending=[True, False]).reset_index(drop=True)

    # QA rápido por consola
    vc = full["mood"].value_counts()
    print("=== Resumen Ejecutivo ===")
    print("\nCandidatas por mood:")
    print(vc)
    for mood in full["mood"].unique():
        samp = full[full["mood"] == mood].head(3)
        tops_str = "; ".join(f"{r.title} ({int(r.release_year) if pd.notna(r.release_year) else 'NA'})"
                             for _, r in samp.iterrows())
        print(f" - {mood}: {tops_str}")

    # Muestra un sample “1 por saga” SOLO para QA, no se guarda capado
    if dedup_saga_for_QA and "saga_key" in full.columns:
        demo = (full.sort_values(["mood", "score_total"], ascending=[True, False])
                    .drop_duplicates(subset=["mood", "saga_key"], keep="first"))
        print("\n[QA] Sample 1 por saga (NO es lo que se guarda):")
        print(demo.groupby("mood").size())

    # Persistencia
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    full.to_parquet(out_path, index=False)
    print(f"\nGuardado OK → {out_path.resolve()}")
    return full

In [117]:
# =========================
# EXPLICABILIDAD (texto por fila) — versión robusta y compatible DF/dict
# =========================

def _band(val, hi=0.75, lo=0.35):
    """Devuelve 'alta', 'media', 'baja' según umbrales en [0,1]."""
    try:
        v = float(val)
        if np.isnan(v):
            return "desconocida"
    except Exception:
        return "desconocida"
    return "alta" if v >= hi else ("media" if v >= lo else "baja")

def _fmt_int(n):
    try:
        n = int(n)
        return f"{n/1000:.1f}k" if n >= 1000 else f"{n}"
    except Exception:
        return "—"

def _runtime_tag(rt):
    try:
        rt = int(rt)
        if rt < 90:   return f"{rt} min (corta)"
        if rt <= 120: return f"{rt} min (estándar)"
        return f"{rt} min (larga)"
    except Exception:
        return "s/d"

def _genres_str(gs, max_items=3):
    # list/tuple/set/ndarray → lista
    if isinstance(gs, list):
        s = ", ".join([str(g) for g in gs[:max_items]])
        return s if s else "s/d"
    try:
        if isinstance(gs, (set, tuple)) or (hasattr(np, "ndarray") and isinstance(gs, np.ndarray)):
            gs = list(gs)
            s = ", ".join([str(g) for g in gs[:max_items]])
            return s if s else "s/d"
    except Exception:
        pass
    return str(gs) if (gs is not None and (not isinstance(gs, float) or not np.isnan(gs))) else "s/d"

def explain_row(row, mood=None):
    """
    Construye una explicación breve usando las columnas del pipeline.
    Espera: match_norm, recencia, rating_norm, votes_norm, pop_norm,
            vote_average, vote_count, release_year, genres, runtime.
    """
    mood = (mood or row.get("mood") or "mood")

    # bandas cualitativas
    b_match = _band(row.get("match_norm"))
    b_rec   = _band(row.get("recencia"))
    b_rat   = _band(row.get("rating_norm"))
    b_vot   = _band(row.get("votes_norm"))
    b_pop   = _band(row.get("pop_norm"))

    # detalles numéricos
    ya   = row.get("release_year", "s/d")
    rt   = _runtime_tag(row.get("runtime"))
    rat  = "s/d"
    try:
        va = row.get("vote_average")
        if va is not None and not (isinstance(va, float) and np.isnan(va)):
            rat = f"{float(va):.1f}"
    except Exception:
        pass
    vcts = _fmt_int(row.get("vote_count"))
    gens = _genres_str(row.get("genres"))

    # hook por mood
    mood_hook = {
        "accion_thriller":    "alineación fuerte con Acción/Thriller",
        "drama_romance":      "alineación fuerte con Drama/Romance",
        "cozy_ligera":        "vibes cozy/ligeras bien marcadas",
        "suspenso_misterio":  "tensión y misterio bien capturados",
    }.get(str(mood), f"alineación con {mood}")

    parts = [
        f"{mood_hook} (match {b_match}).",
        f"Año {ya}, {rt}.",
        f"Rating {rat} con {vcts} votos (confianza {b_vot}).",
        f"Recencia {b_rec}, popularidad {b_pop}.",
        f"Géneros: {gens}."
    ]

    # tags extra
    try:
        if float(row.get("rating_norm", 0)) >= 0.75 and float(row.get("votes_norm", 0)) < 0.45:
            parts.append("⚑ Posible joya oculta (buen rating con pocos votos).")
        if float(row.get("pop_norm", 0)) >= 0.75 and float(row.get("rating_norm", 0)) < 0.55:
            parts.append("⚑ Crowd-pleaser (muy popular, rating medio).")
        if float(row.get("recencia", 0)) >= 0.75:
            parts.append("⚑ Relativamente reciente.")
    except Exception:
        pass

    return " ".join(parts)

def add_explanations_to_df(df: pd.DataFrame) -> pd.DataFrame:
    """Devuelve una copia de df con columna 'explicacion' y la coloca tras score_total."""
    if df is None or df.empty:
        return df
    df2 = df.copy()
    if "mood" not in df2.columns:
        df2["mood"] = None
    df2["explicacion"] = df2.apply(lambda r: explain_row(r, r.get("mood")), axis=1)

    # coloca 'explicacion' cerca de score_total
    cols = list(df2.columns)
    if "explicacion" in cols:
        cols.remove("explicacion")
        insert_at = cols.index("score_total") + 1 if "score_total" in cols else len(cols)
        cols = cols[:insert_at] + ["explicacion"] + cols[insert_at:]
        df2 = df2[cols]
    return df2

# ===== Aplicación al consolidado y guardado =====

df_all_exp = add_explanations_to_df(df_tops)

# Persistencia SIN recortes (parquet + csv)
Path("data").mkdir(parents=True, exist_ok=True)
df_all_exp.to_parquet("data/tops.parquet", index=False)
df_all_exp.to_csv("data/tops.csv", index=False)

# QA rápido en salida
print("Guardado tops.parquet y tops.csv (sin recortes).")
print(df_all_exp.groupby("mood").size().sort_index())

df_all_exp.head(5)

Guardado tops.parquet y tops.csv (sin recortes).
mood
accion_thriller      2571
cozy_ligera          5492
drama_romance        3955
suspenso_misterio    1938
dtype: int64


,title,release_year,genres,runtime,vote_average,vote_count,popularity,poster_url_w342,overview,saga_key,match_raw,match_norm,recencia,rating_norm,votes_norm,pop_norm,score_total,explicacion,rank_within_mood,mood
0,Call Me by Your Name,2017,"[Romance, Drama]",122,8.107,12457,9.2528,https://image.tmdb.org/t/p/w342/cfqc9Z287VPRV8...,"Elio Perlman (Timothée Chalamet), un joven de ...",call me by your name collection,4,1.0,0.68,0.827579,0.984198,0.008360,0.814740,alineación fuerte con Drama/Romance (match alt...,1,drama_romance
1,Shutter Island,2010,"[Drama, Suspense, Misterio]",138,8.201,24970,16.8897,https://image.tmdb.org/t/p/w342/oemYX0Do8bxqVH...,Verano de 1954. Los agentes judiciales Teddy D...,shutter island,4,1.0,0.40,0.868904,0.991266,0.058719,0.812225,tensión y misterio bien capturados (match alta...,1,suspenso_misterio
2,"Con amor, Simon",2018,"[Comedia, Drama, Romance]",115,7.986,6154,2.7947,https://image.tmdb.org/t/p/w342/eQGCjxKFdSHBek...,Simon Spier es un joven 16 años que no se atre...,"con amor, simon",4,1.0,0.72,0.772081,0.968524,0.002471,0.799150,alineación fuerte con Drama/Romance (match alt...,2,drama_romance
3,Perdida,2014,"[Misterio, Suspense, Drama]",149,7.889,19350,18.1501,https://image.tmdb.org/t/p/w342/bkIhygnbv9ydlD...,"El día de su quinto aniversario de boda, Nick ...",perdida,4,1.0,0.56,0.742744,0.988758,0.063214,0.796883,tensión y misterio bien capturados (match alta...,2,suspenso_misterio
4,El arte de vivir bajo la lluvia,2019,"[Drama, Romance]",110,8.188,1591,4.4060,https://image.tmdb.org/t/p/w342/jqeLKlKpaS4v89...,El argumento está basado en la novela publicad...,el arte de vivir bajo la lluvia,3,1.0,0.76,0.826907,0.913843,0.003940,0.774702,vibes cozy/ligeras bien marcadas (match alta)....,1,cozy_ligera
